In [1]:
# ============================================================
# EXPERIMENT 7: CONSTRAINT SPECIFICATION AND ITERATIVE REFINEMENT
# Task: Generate a short description of a smartwatch
# Groq API + Google Colab
# ============================================================

!pip install -q groq

import os
import re
from getpass import getpass
from groq import Groq

# ------------------------------------------------------------
# STEP 1: Load Groq API Key
# ------------------------------------------------------------

groq_api_key = getpass("Enter your Groq API key: ")

if not groq_api_key:
    raise ValueError("GROQ_API_KEY not provided.")

os.environ["GROQ_API_KEY"] = groq_api_key

print("✅ Groq API key loaded successfully.")


# ------------------------------------------------------------
# STEP 2: Create Groq Client
# ------------------------------------------------------------

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

print("✅ Groq client initialized.")


# ------------------------------------------------------------
# STEP 3: Define LLM Function
# ------------------------------------------------------------

def ask_llm(prompt):

    try:

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            temperature=0.5,
            max_tokens=150,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        return response.choices[0].message.content.strip()

    except Exception as e:

        return f"ERROR: {e}"


# ============================================================
# STEP 4: Initial Prompt
# ============================================================

initial_prompt = """
Describe a smartwatch in a maximum of 60 words.

Constraints:
1. Use exactly 3 bullet points.
2. Each bullet point must start with a verb.
3. Do not use the word "smart" anywhere.
"""


# ============================================================
# STEP 5: Send Initial Prompt
# ============================================================

initial_response = ask_llm(initial_prompt)


# ============================================================
# STEP 6: Constraint Checking Function
# ============================================================

def check_constraints(response):

    violations = []

    # --------------------------------------------------------
    # Check word count
    # --------------------------------------------------------

    words = response.split()

    if len(words) > 60:
        violations.append(
            f"Word count violation: {len(words)} words"
        )

    # --------------------------------------------------------
    # Extract bullet points
    # --------------------------------------------------------

    lines = response.splitlines()

    bullet_lines = []

    for line in lines:

        line = line.strip()

        if line.startswith("-"):
            bullet_lines.append(line)

    # --------------------------------------------------------
    # Check exactly 3 bullets
    # --------------------------------------------------------

    if len(bullet_lines) != 3:
        violations.append(
            f"Bullet count violation: found {len(bullet_lines)}, "
            f"expected 3"
        )

    # --------------------------------------------------------
    # Check banned word "smart"
    # --------------------------------------------------------

    if re.search(r"\bsmart\b", response, re.IGNORECASE):

        violations.append(
            'Banned word violation: "smart" was used'
        )

    # --------------------------------------------------------
    # Check whether each bullet starts with a verb
    # --------------------------------------------------------

    # Common verbs that may be used to describe a watch.
    common_verbs = [
        "track",
        "tracks",
        "monitor",
        "monitors",
        "display",
        "displays",
        "measure",
        "measures",
        "record",
        "records",
        "provide",
        "provides",
        "connect",
        "connects",
        "support",
        "supports",
        "show",
        "shows",
        "offer",
        "offers",
        "manage",
        "manages",
        "count",
        "counts",
        "detect",
        "detects",
        "send",
        "sends",
        "receive",
        "receives",
        "control",
        "controls",
        "track",
        "help",
        "helps",
        "lasts"
    ]

    if len(bullet_lines) == 3:

        for bullet in bullet_lines:

            # Remove "-" and spaces
            content = bullet[1:].strip()

            if not content:
                violations.append(
                    "Empty bullet point found."
                )
                continue

            first_word = content.split()[0].lower()

            # Remove punctuation
            first_word = re.sub(
                r"[^a-zA-Z]",
                "",
                first_word
            )

            if first_word not in common_verbs:

                violations.append(
                    f'Bullet does not appear to start with a verb: '
                    f'"{content}"'
                )

    return violations


# ============================================================
# STEP 7: Analyze Initial Response
# ============================================================

initial_violations = check_constraints(
    initial_response
)


# ============================================================
# STEP 8: Display Initial Result
# ============================================================

print("\n" + "=" * 70)
print("INITIAL PROMPT")
print("=" * 70)

print("\nPrompt:")
print(initial_prompt)

print("\nResponse:")
print(initial_response)

print("\nViolation Report:")

if initial_violations:

    for violation in initial_violations:
        print("❌", violation)

else:

    print("✅ No constraint violations detected.")


# ============================================================
# STEP 9: Refined Prompt
# ============================================================

refined_prompt = """
Describe a fitness watch in no more than 60 words.

Follow these instructions exactly:

- Output exactly 3 bullet points.
- Each bullet must begin with an action verb such as
  "Tracks", "Displays", or "Monitors".
- Never use the word "smart", including inside another sentence.
- Do not add an introduction, conclusion, heading, or extra text.

Example format:
- Tracks daily activity.
- Displays useful notifications.
- Monitors exercise sessions.

Now provide a new description using exactly 3 bullet points.
"""


# ============================================================
# STEP 10: Send Refined Prompt
# ============================================================

refined_response = ask_llm(
    refined_prompt
)


# ============================================================
# STEP 11: Check Refined Response
# ============================================================

refined_violations = check_constraints(
    refined_response
)


# ============================================================
# STEP 12: Display Refined Result
# ============================================================

print("\n" + "=" * 70)
print("REFINED PROMPT")
print("=" * 70)

print("\nPrompt:")
print(refined_prompt)

print("\nResponse:")
print(refined_response)

print("\nViolation Report:")

if refined_violations:

    for violation in refined_violations:
        print("❌", violation)

else:

    print("✅ No constraint violations detected.")


# ============================================================
# STEP 13: Compare Results
# ============================================================

print("\n" + "=" * 70)
print("COMPARISON")
print("=" * 70)

initial_count = len(initial_violations)
refined_count = len(refined_violations)

print(
    f"\nInitial prompt violations : {initial_count}"
)

print(
    f"Refined prompt violations : {refined_count}"
)

if refined_count < initial_count:

    print(
        "\n✅ Refinement improved constraint adherence."
    )

elif refined_count == initial_count:

    print(
        "\n⚠️ Refinement produced the same number of violations."
    )

else:

    print(
        "\n❌ Refinement increased the number of violations."
    )


# ============================================================
# STEP 14: Final Analysis
# ============================================================

print("\n" + "=" * 70)
print("FINAL ANALYSIS")
print("=" * 70)

print("""
Initial Prompt:
The initial prompt specifies the required constraints, but the
instructions may be interpreted differently by the model.

Refined Prompt:
The refined prompt makes the constraints more explicit, gives
examples of acceptable bullet beginnings, and tells the model
not to include additional text.

Iterative Refinement:
The prompt was improved without changing the core task.
The goal was to make the required format and restrictions clearer.

Conclusion:
Clear constraints, examples, and explicit formatting instructions
can improve an LLM's ability to follow multiple requirements.
""")

print("✅ Constraint specification experiment completed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.6 MB/s eta 0:00:00
Enter your Groq API key: ··········
✅ Groq API key loaded successfully.
✅ Groq client initialized.

INITIAL PROMPT

Prompt:

Describe a smartwatch in a maximum of 60 words.

Constraints:
1. Use exactly 3 bullet points.
2. Each bullet point must start with a verb.
3. Do not use the word "smart" anywhere.


Response:
* Tracking fitness goals
* Monitoring heart rate
* Displaying notifications

Violation Report:
❌ Bullet count violation: found 0, expected 3

REFINED PROMPT

Prompt:

Describe a fitness watch in no more than 60 words.

Follow these instructions exactly:

- Output exactly 3 bullet points.
- Each bullet must begin with an action verb such as
  "Tracks", "Displays", or "Monitors".
- Never use the word "smart", including inside another sentence.
- Do not add an introduction, conclusion, heading, or extra text.

Example format:
- Tracks daily activity.
- Displays useful notifications.
- Monitors exerc